In [0]:
# 1. Definimos origen y destino
esquema_origen = "dev_practica_ui.bronze"
esquema_destino = "comercializadora.sistema_ventas"

# 2. Aseguramos que el esquema de destino exista
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {esquema_destino}")

# 3. Obtenemos la lista de todas las tablas del origen
tablas = [row.tableName for row in spark.sql(f"SHOW TABLES IN {esquema_origen}").collect() if not row.isTemporary]

# 4. Clonamos cada tabla automáticamente
for tabla in tablas:
    print(f"Clonando {esquema_origen}.{tabla} -> {esquema_destino}.{tabla}...")
    spark.sql(f"""
        CREATE OR REPLACE TABLE {esquema_destino}.{tabla}
        DEEP CLONE {esquema_origen}.{tabla}
    """)

print("¡Todas las tablas se han clonado exitosamente en comercializadora.sistema_ventas!")

In [0]:
# 1. Definimos origen y destino
esquema_origen = "dev_practica_ui.bronze"
esquema_destino = "comercializadora.sistema_ventas"

# 2. Aseguramos que el esquema de destino exista
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {esquema_destino}")

# 3. Obtenemos el detalle de los objetos en el origen
objetos = spark.sql(f"SHOW TABLES IN {esquema_origen}").collect()

for obj in objetos:
    nombre = obj.tableName
    
    # Revisamos si es una Vista o una Tabla
    es_vista = spark.sql(f"DESCRIBE DETAIL {esquema_origen}.{nombre}").select("format").collect()[0]["format"] == "view"
    
    if es_vista:
        print(f"Recreando Vista: {esquema_origen}.{nombre} -> {esquema_destino}.{nombre}...")
        # Obtenemos el DDL original de la vista
        ddl_vista = spark.sql(f"SHOW CREATE TABLE {esquema_origen}.{nombre}").collect()[0][0]
        # Reemplazamos la referencia del esquema en el DDL
        ddl_destino = ddl_vista.replace(esquema_origen, esquema_destino)
        spark.sql(ddl_destino)
    else:
        print(f"Clonando Tabla: {esquema_origen}.{nombre} -> {esquema_destino}.{nombre}...")
        spark.sql(f"""
            CREATE OR REPLACE TABLE {esquema_destino}.{nombre}
            DEEP CLONE {esquema_origen}.{nombre}
        """)

print("¡Proceso completado exitosamente! Tablas clonadas y vistas recreadas.")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS comercializadora.sistema_ventas;

CREATE OR REPLACE TABLE comercializadora.sistema_ventas.clientes DEEP CLONE dev_practica_ui.bronze.clientes;
CREATE OR REPLACE TABLE comercializadora.sistema_ventas.cumplimiento_metas DEEP CLONE dev_practica_ui.bronze.cumplimiento_metas;
CREATE OR REPLACE TABLE comercializadora.sistema_ventas.productos DEEP CLONE dev_practica_ui.bronze.productos;
CREATE OR REPLACE TABLE comercializadora.sistema_ventas.vendedores DEEP CLONE dev_practica_ui.bronze.vendedores;
CREATE OR REPLACE TABLE comercializadora.sistema_ventas.ventas DEEP CLONE dev_practica_ui.bronze.ventas;
CREATE OR REPLACE TABLE comercializadora.sistema_ventas.ventas_por_ciudad_mes DEEP CLONE dev_practica_ui.bronze.ventas_por_ciudad_mes;

In [0]:
%sql
CREATE OR REPLACE VIEW comercializadora.sistema_ventas.v_ventas_enriquecidas (
  id_venta COMMENT 'Identificador único de la venta',
  fecha_venta,
  id_cliente COMMENT 'Cliente que compró (clientes.id_cliente)',
  id_vendedor COMMENT 'Vendedor (vendedores.id_vendedor). NULL en ventas en línea',
  id_producto COMMENT 'Producto vendido (productos.id_producto)',
  canal COMMENT 'Tienda física, En línea o Teléfono',
  estado COMMENT 'Completada, Devuelta o Anulada',
  cantidad COMMENT 'Unidades vendidas',
  precio_unitario COMMENT 'Precio por unidad al momento de la venta, en COP',
  descuento_pct,
  valor_neto,
  metodo_pago COMMENT 'Medio de pago (campo dentro de struct pago)',
  cuotas COMMENT 'Número de cuotas (campo dentro de struct pago)',
  nombre_cliente COMMENT 'Nombre completo del cliente',
  email_cliente COMMENT 'Correo electrónico de contacto',
  ciudad_cliente COMMENT 'Ciudad de residencia del cliente',
  nombre_vendedor,
  sede_vendedor,
  nombre_producto COMMENT 'Nombre comercial del producto',
  categoria_producto COMMENT 'Tecnología, Hogar, Oficina o Deportes',
  costo_unitario COMMENT 'Costo de compra por unidad, en COP',
  utilidad)
DEFAULT COLLATION UTF8_BINARY
WITH SCHEMA COMPENSATION
AS WITH cte_base AS (
    SELECT v.id_venta
         , CAST(v.fecha_venta AS DATE) AS fecha_venta
         , v.id_cliente
         , v.id_vendedor
         , v.id_producto
         , v.canal
         , v.estado
         , v.cantidad
         , v.precio_unitario
         , COALESCE(v.descuento_pct, 0) AS descuento_pct
         , (v.cantidad * v.precio_unitario * (1 - COALESCE(v.descuento_pct, 0))) AS valor_neto
         , v.pago.metodo AS metodo_pago
         , v.pago.cuotas AS cuotas
    FROM dev_practica_ui.bronze.ventas v
)
SELECT b.id_venta
     , b.fecha_venta
     , b.id_cliente
     , b.id_vendedor
     , b.id_producto
     , b.canal
     , b.estado
     , b.cantidad
     , b.precio_unitario
     , b.descuento_pct
     , CAST(b.valor_neto AS DECIMAL(14,2)) AS valor_neto
     , b.metodo_pago
     , b.cuotas
     -- Datos del Cliente
     , c.nombre_cliente
     , c.email AS email_cliente
     , c.ciudad AS ciudad_cliente
     -- Datos del Vendedor
     , vend.nombre_vendedor
     , vend.sede AS sede_vendedor
     -- Datos del Producto
     , p.nombre_producto
     , p.categoria AS categoria_producto
     , p.costo_unitario
     -- Cálculo de Utilidad
     , CAST(b.valor_neto - (b.cantidad * p.costo_unitario) AS DECIMAL(14,2)) AS utilidad
FROM cte_base b
LEFT JOIN dev_practica_ui.bronze.clientes c
       ON b.id_cliente = c.id_cliente
LEFT JOIN dev_practica_ui.bronze.vendedores vend
       ON b.id_vendedor = vend.id_vendedor
LEFT JOIN dev_practica_ui.bronze.productos p
       ON b.id_producto = p.id_producto -- (O la consulta SQL interna de tu vista original)comercializadora.sistema_ventas.v_ventas_enriquecidascomercializadora.sistema_ventas.v_ventas_enriquecidas

In [0]:
%sql
SELECT * FROM comercializadora.sistema_ventas.ventas; 

In [0]:
%sql
SELECT * FROM dev_practica_ui.bronze.ventas; 

In [0]:
%sql
SHOW CREATE TABLE dev_practica_ui.bronze.v_ventas_enriquecidas;